In [0]:
from pyspark.sql.functions import *


In [0]:
silver_df = spark.table("workspace.bronze.orders")

In [0]:
silver_df.printSchema()

In [0]:
from pyspark.sql.functions import round

silver_df = silver_df.withColumn(
    "Sales",
    round(col("Sales"), 4)
)

silver_df = silver_df.withColumn(
    "Profit",
    round(col("Profit"), 4)
)

In [0]:
print(silver_df.schema)

In [0]:
from pyspark.sql.functions import to_date, col

silver_df = silver_df.withColumn(
    "Order Date",
    to_date(col("Order Date"))
)

silver_df = silver_df.withColumn(
    "Ship Date",
    to_date(col("Ship Date"))
)

In [0]:
import re

In [0]:
def to_snake_case(column_name):
    column_name = column_name.lower()
    column_name = re.sub(r'[^a-z0-9]+', '_', column_name)
    column_name = column_name.strip('_')
    return column_name

In [0]:
silver_df = silver_df.toDF(*[to_snake_case(col) for col in silver_df.columns])
display(silver_df)

In [0]:
print(silver_df.columns)

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

for field in silver_df.schema.fields:
    if isinstance(field.dataType, StringType):
        silver_df = silver_df.withColumn(
            field.name,
            trim(col(field.name))
        )

**Check for NULL values**

In [0]:
from pyspark.sql.functions import col, sum, when

null_counts = silver_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in silver_df.columns
])

display(null_counts)

**Check Duplicate Rows**

In [0]:
total_rows = silver_df.count()

distinct_rows = silver_df.distinct().count()

print(f"Total Rows : {total_rows}")
print(f"Distinct Rows : {distinct_rows}")
print(f"Duplicate Rows : {total_rows - distinct_rows}")

**Data Quality Checks**

In [0]:
silver_df.filter(col("sales") < 0).show()

Removeing of Duplicate records

In [0]:
print(f"Rows Before : {silver_df.count()}")

silver_df = silver_df.dropDuplicates()

print(f"Rows After : {silver_df.count()}")

Data Quality Validation 

In [0]:
silver_df.filter(col("quantity") <= 0).show()

--------------------------------------------Order date should ever be after ship date 

In [0]:
silver_df.filter(
    col("order_date") > col("ship_date")
).show()

**Save Silver Table**

In [0]:
silver_df.write \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.superstore_orders_silver")

In [0]:
silver_check = spark.table(
    "workspace.silver.superstore_orders_silver"
)

display(silver_check)